# Stage 7 — Deliberately break the backtest

**Self-directed academic project.** This exercise compares a cost-aware baseline with two intentionally invalid variants: ignoring execution costs and using the signal from the end of a return interval to decide exposure for that same interval. The overfit demonstration also picks SMA parameters on the same observations it reports. None of those biased results is evidence of a tradable edge.

The dashboard asks for a benchmark and drawdown prediction before each run, then compares the guess with the held-out results. It also includes short questions on look-ahead bias, drawdown, survivorship bias, and optimisation uncertainty.

In [ ]:
import numpy as np
import pandas as pd

from portfolio_research.bias_lab import compare_signal_biases
from portfolio_research.config import SmaSettings
from portfolio_research.strategies import sma_signals

index = pd.bdate_range("2020-01-01", periods=700)
rng = np.random.default_rng(17)
prices = pd.Series(
    100 * (1 + rng.normal(0.0003, 0.012, len(index))).cumprod(), index=index
)
settings = SmaSettings(20, 80, 5, 5, 1)
signals = sma_signals(prices, settings.fast_window, settings.slow_window)
comparison = compare_signal_biases(prices, signals, settings)
pd.DataFrame(
    {
        "honest net returns": comparison.baseline.net_returns,
        "costs ignored": comparison.costs_ignored.net_returns,
        "future signal leaked into return period": comparison.future_signal_used.net_returns,
    }
).tail()

Compare full-period cumulative wealth only to understand the direction of the distortions. The exact effect depends on the sample; look-ahead bias is invalid even when it makes results worse. A robust research conclusion is not that one biased number is larger, but that the assumptions changed and only the causal, cost-aware version is eligible for evaluation.